# Fig. S7: time evolution of S2, Q and S_half

S2(t), Q(t) and S_half(t) for the initial Fock states of Fig. 3 (a-c) and
Fig. S3 (d-f), with the same columns and colours, evolved exactly in each
sector on 2001 points over Jt in [0, 400]. S_half uses the cuts of Fig. S5.

Data: the Fig. 3 and Fig. S3 example files.

In [ ]:
%matplotlib inline

import h5py
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

from qising_networks import DATA_DIR as RESULTS, FIGURES_DIR as FIGURES, PROJECT_ROOT
from qising_networks import fig_maker as fm
from qising_networks import fig_maker_helper as fmh
from qising_networks import scar_metrics as sm
from qising_networks import spectra
from qising_networks.model import sector_hamiltonian
from qising_networks.plots.common import CLASS_COLORS
from qising_networks.scar_metrics import DISPLAY_TIME_MAX
from qising_networks.plots.dynamics import (CLASS_ORDER, MODELS,
                                            ring_thermal_column)

mpl.rc_file(PROJECT_ROOT / "matplotlibrc")
N_TIME = 2001

In [ ]:
def block_columns(model, net_file, ring_file):
    """The 7 columns of Fig. 3 (or S3): (class, im, h, sec_idx, H, alpha)."""
    with h5py.File(net_file) as f:
        im = f['im'][:]
        h = f['h'][:] if 'h' in f else f['hz'][:]
        sec_idx = f['sec_idx'][:]
        alphas = f['alphas'][:]
    H = sector_hamiltonian(im, h, sec_idx)
    cols = [dict(cls=c, sec_idx=sec_idx, H=H, alpha=int(alphas[c]),
                 N=len(h)) for c in CLASS_ORDER]
    with h5py.File(ring_file) as f:
        rim = f['im'][:]
        rh = f['h'][:] if 'h' in f else f['hz'][:]
        g = f['class_2']  # Neel scar
        rsec = g['sec_idx'][:]
        ralpha = int(g.attrs['alpha'])
    rH = sector_hamiltonian(rim, rh, rsec)
    cols.append(dict(cls=int(sm.SCAR),
                     sec_idx=rsec, H=rH, alpha=ralpha, N=len(rh)))
    # the thermalized re-pick of Figs. 3 and S3
    cols.append(dict(cls=int(sm.THERMALIZED), sec_idx=rsec, H=rH,
                     alpha=ring_thermal_column(model)['alpha'], N=len(rh)))
    return cols


# Half-system cuts of Fig. S5: the median balanced bipartition for the
# networks, the N/2 contiguous half-ring cuts for the rings.
ER_CUTS = {'ferro': [0, 1, 2, 8, 9, 10], 'pmJ': [0, 7, 8, 9, 10, 11]}


def evolve_s2_q(col, time):
    """S2(t), Q(t), S_half(t) of exp(-iHt)|s_alpha> within the column's
    sector."""
    E, V = np.linalg.eigh(col['H'])
    c0 = V[col['alpha'], :]
    psi = V @ (c0[:, None] * np.exp(-1j * np.outer(E, time)))  # (dim, T)
    s2 = spectra.participation_entropy(psi, order=2)
    q = spectra.global_entanglement(
        psi, col['sec_idx'], col['N'])
    sh = np.mean([spectra.half_entropy(psi, col['sec_idx'], col['N'], A)
                  for A in col['cuts']], 0)
    return s2, q, sh


time = np.linspace(0.0, DISPLAY_TIME_MAX, N_TIME)
blocks = {model: block_columns(model, RESULTS / MODELS[model]['net_file'],
                               RESULTS / MODELS[model]['ring_file'])
          for model in ('ferro', 'pmJ')}
for name, cols in blocks.items():
    for k, col in enumerate(cols):
        N = col['N']
        col['cuts'] = ([ER_CUTS[name]] if k < len(CLASS_ORDER) else
                       [[(j + i) % N for i in range(N // 2)]
                        for j in range(N // 2)])
        col['s2'], col['q'], col['sh'] = evolve_s2_q(col, time)
        print(f"{name} {fmh.CLASS_NAMES[col['cls']]:>13}: "
              f"dim={len(col['sec_idx'])} alpha={col['alpha']} "
              f"S2(0)={col['s2'][0]:.2e} Q(0)={col['q'][0]:.2e} "
              f"late mean S2={col['s2'][time > 200].mean():.2f} "
              f"Q={col['q'][time > 200].mean():.2f} "
              f"S_half={col['sh'][time > 200].mean():.2f} "
              f"(min {col['sh'][time > 200].min():.2f}, "
              f"max {col['sh'][time > 200].max():.2f}; "
              f"first 1/2 crossing Jt="
              f"{time[np.argmax(col['sh'] > 0.5 * col['sh'][time > 200].mean())]:.1f})")
    # the ring's Neel scar relative to its thermalized state, late times
    scar, therm = cols[-2], cols[-1]
    print(f"{name} ring scar / thermalized: "
          f"S_half {scar['sh'][time > 200].mean() / therm['sh'][time > 200].mean():.2f}, "
          f"Q {scar['q'][time > 200].mean() / therm['q'][time > 200].mean():.2f}")


In [ ]:
N_ER = 5
BLOCKS = (
    ('ferro', 'Ferromagnetic model',
     ('Erdős–Rényi network', r'1D ring, uniform $h = 2J$')),
    ('pmJ', r'$\pm J$ model',
     (r'Erdős–Rényi network ($\pm J$)', r'Frustration-free $\pm J$ ring')),
)

with plt.rc_context({'font.size': 7}):
    fig = plt.figure(figsize=(fm.MAX_FIG_WIDTH, 6.6), constrained_layout=True)
    fig.get_layout_engine().set(h_pad=0.02, w_pad=0.01, hspace=0.02,
                                wspace=0.02)
    sf_blocks = fig.subfigures(2, 1, hspace=0.04)
    row_first_axes = []
    # One grid per block: a header row, then S2, Q and S_half; column 5 is a
    # spacer between network and ring.
    data_cols = list(range(N_ER)) + [N_ER + 1, N_ER + 2]
    for sf_block, (key, block_title, sub_titles) in zip(sf_blocks, BLOCKS):
        sf_block.suptitle(block_title, fontsize=7, fontweight='bold')
        gs = sf_block.add_gridspec(
            4, N_ER + 3, height_ratios=[0.10, 1, 1, 1],
            width_ratios=[1] * N_ER + [0.12, 1, 1])
        for span, sub_title in ((slice(0, N_ER), sub_titles[0]),
                                (slice(N_ER + 1, N_ER + 3), sub_titles[1])):
            ax_h = sf_block.add_subplot(gs[0, span])
            ax_h.axis('off')
            ax_h.text(0.5, 0.0, sub_title, ha='center', va='bottom',
                      transform=ax_h.transAxes, fontsize=7)
        cols = blocks[key]
        ax_s0 = ax_q0 = ax_e0 = None
        for gc, col in zip(data_cols, cols):
            if ax_s0 is None:
                ax_s = ax_s0 = sf_block.add_subplot(gs[1, gc])
                ax_q = ax_q0 = sf_block.add_subplot(gs[2, gc], sharex=ax_s)
                ax_e = ax_e0 = sf_block.add_subplot(gs[3, gc], sharex=ax_s)
            else:
                ax_s = sf_block.add_subplot(gs[1, gc], sharey=ax_s0)
                ax_q = sf_block.add_subplot(gs[2, gc], sharex=ax_s,
                                            sharey=ax_q0)
                ax_e = sf_block.add_subplot(gs[3, gc], sharex=ax_s,
                                            sharey=ax_e0)
                ax_s.tick_params(labelleft=False)
                ax_q.tick_params(labelleft=False)
                ax_e.tick_params(labelleft=False)
            colour = CLASS_COLORS[col['cls']]
            # Above the spines: Q = S_half = 0 for the localized and recurrence states.
            line = dict(lw=0.7, color=colour, zorder=3, clip_on=False)
            ax_s.plot(time, col['s2'], **line)
            ax_q.plot(time, col['q'], **line)
            ax_e.plot(time, col['sh'], **line)
            ax_s.set_title(fmh.CLASS_NAMES[col['cls']], color=colour, pad=2)
            ax_e.set_xlim(time[0], time[-1])
            ax_e.set_xticks([0, 200, 400])
            ax_e.set_xlabel(r'Time $Jt$')
            ax_s.tick_params(labelbottom=False)
            ax_q.tick_params(labelbottom=False)
        ax_s0.set_ylabel(r'$S_2(t)$')
        ax_q0.set_ylabel(r'$Q(t)$')
        ax_s0.set_ylim(bottom=0)
        ax_q0.set_ylim(0, 1.05)
        ax_e0.set_ylabel(r'$S_\mathrm{half}(t)$')
        ax_e0.set_ylim(bottom=0)
        row_first_axes += [ax_s0, ax_q0, ax_e0]

    fm.align_panel_labels_to_yaxis(
        fig, row_first_axes, fm.journal_labels("abcdef"), fontsize=8,
        bold=True)
    # Align the labels to the leftmost one.
    label_texts = [ax.texts[-1] for ax in row_first_axes]
    x_left = min(ax.transAxes.transform(t.get_position())[0]
                 for ax, t in zip(row_first_axes, label_texts))
    for ax, t in zip(row_first_axes, label_texts):
        x_ax = ax.transAxes.inverted().transform((x_left, 0))[0]
        t.set_x(x_ax)


In [ ]:
paths = fm.save_unlabeled(fig, FIGURES / 'fig-s07', 'fig-s07')
print(paths)
